In [19]:
import gzip, pickle, math, pandas as pd, numpy as np

# ---------------- Parameters ----------------------
PHASE2_PKL_GZ   = "phase2_clusters.pkl.gz"
TRAJ_PKL_GZ     = "phase1_trajectories.pkl.gz"   # trip paths if you need finer distance later
VEHICLES_CSV    = "vehicles.csv"
RIDES_CSV       = "Taxi_Trips__2024-__20250611.csv"

MIN_BATTERY_PCT = 30
EV_RANGE_KM     = 160              # assume full range
EV_KM_PER_PCT   = EV_RANGE_KM / 100
BBOX_DEG        = 0.1
MAX_TRIP_KM     = 20               # skip clusters whose trip pickup distance > 20 km

# ---------------- Helpers -------------------------
RAD = math.pi / 180

def haversine_np(lat1, lon1, lats2, lons2):
    lat1 = lat1 * RAD; lon1 = lon1 * RAD
    lats2 = lats2 * RAD; lons2 = lons2 * RAD
    dlat = lats2 - lat1; dlon = lons2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lats2)*np.sin(dlon/2)**2
    return 6371000 * 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))  # metres

# ---------------- Load Data -----------------------
with gzip.open(PHASE2_PKL_GZ, "rb") as f:
    clusters = pickle.load(f)
print("Clusters:", len(clusters))

veh = pd.read_csv(VEHICLES_CSV).rename(columns={
    "Vehicle_ID": "vehicle_id", "Type": "type",
    "Current_Latitude": "lat", "Current_Longitude": "lon",
    "Battery_Percentage": "battery", "Status": "status"
})
veh["status"] = veh["status"].str.strip().str.lower()
veh["type"]   = veh["type"].str.strip().str.upper()

idle_mask = (veh.status == "idle") & veh.lat.notna() & veh.lon.notna()
veh_idle = veh[idle_mask].copy().reset_index(drop=True)
veh_idle["taken"] = False
print("Idle vehicles:", len(veh_idle))

rides_df = pd.read_csv(
    RIDES_CSV,
    usecols=[
        "Trip_ID", "Pickup_Centroid_Latitude", "Pickup_Centroid_Longitude",
        "Dropoff_Centroid_Latitude", "Dropoff_Centroid_Longitude"
    ]
).rename(columns={
    "Trip_ID": "trip_id", "Pickup_Centroid_Latitude": "pickup_lat",
    "Pickup_Centroid_Longitude": "pickup_lon", "Dropoff_Centroid_Latitude": "drop_lat",
    "Dropoff_Centroid_Longitude": "drop_lon"
}).dropna()

# compute trip distance in km
rides_df["trip_km"] = haversine_np(
    rides_df.pickup_lat.values,
    rides_df.pickup_lon.values,
    rides_df.drop_lat.values,
    rides_df.drop_lon.values
) / 1000

rides_lookup = rides_df.set_index("trip_id")

# masks by type
mask_ev     = (veh_idle.type == "EV") & (veh_idle.battery >= MIN_BATTERY_PCT)
mask_petrol = (veh_idle.type == "PETROL")
mask_diesel = (veh_idle.type == "DIESEL")
base_masks = {"EV": mask_ev, "PETROL": mask_petrol, "DIESEL": mask_diesel}

# ---------------- Assignment Loop -----------------
assign_rows = []
counts = {"bad_id":0, "too_long":0, "assigned":0, "battery_skip":0, "no_idle":0}

for cl in clusters:
    first_trip = cl["members"][0]["trip_id"]
    if first_trip not in rides_lookup.index:
        counts["bad_id"] += 1
        assign_rows.append({"cluster_id": cl["cluster_id"], "vehicle_id": None})
        continue

    rec = rides_lookup.loc[first_trip]
    trip_km  = rec.trip_km
    if trip_km > MAX_TRIP_KM:
        counts["too_long"] += 1
        assign_rows.append({"cluster_id": cl["cluster_id"], "vehicle_id": None})
        continue

    pick_lat, pick_lon = rec.pickup_lat, rec.pickup_lon
    assigned = False

    for vtype in ("EV", "PETROL", "DIESEL"):
        idxs = np.where(base_masks[vtype] & (~veh_idle.taken.values))[0]
        if idxs.size == 0:
            continue
        lat_diff = np.abs(veh_idle.lat.values[idxs] - pick_lat)
        lon_diff = np.abs(veh_idle.lon.values[idxs] - pick_lon)
        cand_idx = idxs[(lat_diff <= BBOX_DEG) & (lon_diff <= BBOX_DEG)]
        if cand_idx.size == 0:
            continue
        dists = haversine_np(pick_lat, pick_lon,
                             veh_idle.lat.values[cand_idx],
                             veh_idle.lon.values[cand_idx])
        order = np.argsort(dists)
        for loc in cand_idx[order]:
            if vtype == "EV":
                needed_pct = (trip_km + dists[np.where(cand_idx==loc)][0]/1000) / EV_KM_PER_PCT
                if veh_idle.at[loc, "battery"] < needed_pct:
                    counts["battery_skip"] += 1
                    continue
            vid = veh_idle.at[loc, "vehicle_id"]
            assign_rows.append({
                "cluster_id": cl["cluster_id"], "vehicle_id": vid, "vehicle_type": vtype,
                "distance_to_pickup_m": round(float(dists[np.where(cand_idx==loc)][0]),1)
            })
            veh_idle.at[loc, "taken"] = True
            counts["assigned"] += 1
            assigned = True
            break
        if assigned:
            break

    if not assigned:
        counts["no_idle"] += 1
        assign_rows.append({"cluster_id": cl["cluster_id"], "vehicle_id": None,
                            "vehicle_type": None, "distance_to_pickup_m": None})

# ---------------- Save ----------------------------
out_df = pd.DataFrame(assign_rows)
out_df.to_csv("phase3_assignments.csv", index=False)
print("✅ assignments:", len(out_df))
print(counts)


Clusters: 1749
Idle vehicles: 2000
✅ assignments: 1749
{'bad_id': 0, 'too_long': 0, 'assigned': 1375, 'battery_skip': 0, 'no_idle': 374}
